# Notebook 01 — Quiz mode
Tests the multiple-choice quiz pipeline end-to-end.

**What this notebook does:**
1. Loads a model (your choice below)
2. Loads the RAG index you built with `chunker.py` + `embedder.py`
3. Retrieves relevant course chunks for a topic
4. Asks the model to generate a JSON quiz question from that context
5. Lets you answer it and gives feedback

**No frontend — just raw cells. Tweak and re-run freely.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION  ← only section you need to edit
# ═══════════════════════════════════════════════════════════

# Pick ONE model by uncommenting it
MODEL_CHOICE = "cpu_tiny"      # flan-t5-small, ~80 MB, no GPU needed
# MODEL_CHOICE = "small_hf"   # flan-t5-base, ~250 MB, slow on CPU, fast on GPU
# MODEL_CHOICE = "larger_hf"  # TinyLlama-1.1B, needs GPU (fits 3080 easily)
# MODEL_CHOICE = "own_model"  # your trained checkpoint from train.py

# Paths (only used for own_model and RAG)
CHECKPOINT_PATH = "../teaching_assistant/checkpoints/step_005000.pt"
RAG_INDEX_DIR   = "../teaching_assistant/rag_index"

# What topic to quiz on
QUIZ_TOPIC = "dropout regularization"

# How many questions to generate in the loop at the bottom
N_QUESTIONS = 3

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL LOADER — compatible with transformers >= 4.40
#
# text2text-generation was removed from the pipeline registry
# in newer transformers. We use the model + tokenizer directly
# instead — it's the same thing under the hood, just explicit.
# ═══════════════════════════════════════════════════════════
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if MODEL_CHOICE == "cpu_tiny":
    # ── flan-t5-small ──────────────────────────────────────
    # T5 is an encoder-decoder (seq2seq) model.
    # We encode the prompt with the encoder, then let the
    # decoder generate the output autoregressively.
    # ~80 MB, runs in seconds on CPU.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-small")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
    _mdl.eval()
    # Note: T5 stays on CPU (device=-1 was the old pipeline arg)

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "small_hf":
    # ── flan-t5-base ───────────────────────────────────────
    # Same architecture, 3× more parameters. Better at following
    # structured instructions (e.g. "output only JSON").
    # ~250 MB. CPU works but is slow; GPU recommended.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-base")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
    _mdl = _mdl.to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt",
                      truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "larger_hf":
    # ── TinyLlama-1.1B-Chat ────────────────────────────────
    # Decoder-only (same family as GPT). Uses a chat template
    # so the prompt is wrapped in <|user|> / <|assistant|> tags.
    # ~2.2 GB VRAM — fits easily on a 3080 laptop.
    # Swap model_id for "microsoft/phi-2" (2.7B) for better quality.
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    _tok = AutoTokenizer.from_pretrained(model_id)
    _mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        # Apply the chat template so TinyLlama knows this is a user message
        messages  = [{"role": "user", "content": prompt}]
        formatted = _tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = _tok(formatted, return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
            )
        # Decode only the newly generated tokens (skip the prompt)
        new_ids = out_ids[0, inputs["input_ids"].shape[1]:]
        return _tok.decode(new_ids, skip_special_tokens=True)

elif MODEL_CHOICE == "own_model":
    # ── Your trained model from train.py ───────────────────
    # Uses tiktoken gpt2 BPE tokenizer (same as training).
    import sys
    sys.path.insert(0, "../teaching_assistant")
    from rag_pipeline import load_model
    import tiktoken
    _own_model, _own_tok, _own_cfg = load_model(CHECKPOINT_PATH, device)
    _enc = tiktoken.get_encoding("gpt2")

    def generate(prompt, max_new_tokens=200):
        ids = _enc.encode(prompt)
        x   = torch.tensor([ids], dtype=torch.long).to(device)
        out = _own_model.generate(
            x,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_k=50,
            stop_token=_enc.eot_token,
        )
        new_ids = out[0, len(ids):].tolist()
        return _enc.decode(new_ids).replace("<|endoftext|>", "").strip()

print(f"Model ready: {MODEL_CHOICE}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD RAG RETRIEVER
# This uses your existing retriever.py + the HNSW index
# built by embedder.py. Skip this cell if you haven't built
# the index yet — you can still test without RAG below.
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "../teaching_assistant")
sys.path.insert(1, "../../rag")
print("Paths:", sys.path)
retriever = None
try:
    from retriever import Retriever
    retriever = Retriever(index_dir=RAG_INDEX_DIR)
    print("RAG retriever loaded")
except Exception as e:
    print(f"RAG not available ({e}) — will use a hardcoded example context")

In [ ]:
# ═══════════════════════════════════════════════════════════
# QUIZ GENERATION FUNCTION
# The model is asked to output ONLY a JSON object.
# We parse it and validate the structure.
# ═══════════════════════════════════════════════════════════
import json, re

QUIZ_PROMPT_TEMPLATE = """Based on the following text, generate ONE multiple-choice quiz question.
Output ONLY a JSON object — no explanation, no preamble:
{{
  "question": "The question text?",
  "options": ["Correct answer", "Wrong option 1", "Wrong option 2", "Wrong option 3"],
  "correct_index": 0,
  "explanation": "One sentence: why option 0 is correct."
}}

Text:
{context}

Topic to focus on: {topic}"""


def get_context(topic):
    """Retrieve relevant context from the RAG index, or fall back to a placeholder."""
    if retriever is not None:
        chunks = retriever.query(topic, top_k=2)
        return "\n---\n".join(c["text"][:400] for c in chunks)
    else:
        # Fallback: hardcoded example so the notebook still runs without a RAG index
        return ("Dropout is a regularization technique for neural networks where "
                "a fraction of neuron outputs are randomly set to zero during training. "
                "This prevents neurons from co-adapting and forces the network to learn "
                "redundant representations, reducing overfitting.")


def generate_quiz(topic):
    """Generate a quiz question dict for the given topic."""
    context = get_context(topic)
    prompt  = QUIZ_PROMPT_TEMPLATE.format(context=context[:600], topic=topic)
    raw     = generate(prompt, max_new_tokens=350)

    # Try to extract JSON from the output
    # (some models add extra text before/after the JSON object)
    for attempt in [raw, re.search(r'\{.*\}', raw, re.DOTALL)]:
        try:
            text = attempt if isinstance(attempt, str) else attempt.group()
            data = json.loads(text)
            # Validate required fields
            assert "question" in data
            assert isinstance(data.get("options"), list) and len(data["options"]) == 4
            assert data.get("correct_index") in [0, 1, 2, 3]
            return data
        except Exception:
            continue
    print(f"[WARN] Could not parse JSON. Raw output:\n{raw}")
    return None


# Quick test — generate one question and show the raw output
print(f"Generating a question about: '{QUIZ_TOPIC}'...\n")
quiz = generate_quiz(QUIZ_TOPIC)
if quiz:
    print("Parsed OK:")
    print(json.dumps(quiz, indent=2))

In [ ]:
# ═══════════════════════════════════════════════════════════
# INTERACTIVE QUIZ LOOP
# Generates N_QUESTIONS questions and prompts for your answer.
# Works in a terminal-style input/output fashion.
# ═══════════════════════════════════════════════════════════

def run_quiz(topic, n_questions=N_QUESTIONS):
    letters  = ["A", "B", "C", "D"]
    score    = 0

    for q_num in range(1, n_questions + 1):
        print(f"\n{'─'*55}")
        print(f"Question {q_num}/{n_questions}  |  Topic: {topic}")
        print('─'*55)

        quiz = generate_quiz(topic)
        if quiz is None:
            print("Generation failed — skipping.")
            continue

        print(f"\n{quiz['question']}\n")
        for i, (letter, opt) in enumerate(zip(letters, quiz["options"])):
            print(f"  {letter})  {opt}")

        # Get user answer
        while True:
            ans = input("\nYour answer (A/B/C/D): ").strip().upper()
            if ans in letters:
                break
            print("Please type A, B, C, or D.")

        chosen = letters.index(ans)
        correct_idx = quiz["correct_index"]

        if chosen == correct_idx:
            print("\n✓  Correct!")
            score += 1
        else:
            print(f"\n✗  Wrong. Correct answer: {letters[correct_idx]})  {quiz['options'][correct_idx]}")

        print(f"   Explanation: {quiz.get('explanation', '—')}")
        print(f"   Score so far: {score}/{q_num}")

    print(f"\n{'═'*55}")
    print(f"Final score: {score}/{n_questions} ({100*score//n_questions}%)")
    print('═'*55)


run_quiz(QUIZ_TOPIC)

In [ ]:
# ═══════════════════════════════════════════════════════════
# INSPECT: see what context the RAG fetched
# Run this to understand WHY the model generated what it did.
# ═══════════════════════════════════════════════════════════
context = get_context(QUIZ_TOPIC)
print(f"Context retrieved for '{QUIZ_TOPIC}':\n")
print(context[:800])